# # Transfer Learning

In [1]:
import sys
import os

from langchain_community.embeddings.bookend import PATH

# Añadimos tanto la raíz como la carpeta CNN al path
sys.path.append("..")           # Para encontrar 'CNN'
sys.path.append("../CNN")      # Para que los archivos dentro de CNN se encuentren entre ellos

C:\Users\beni7\PycharmProjects\chrome-dino\.env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports


In [5]:
import torch
from CNN import train_utils , models , transforms , data , utils , evaluation
import optuna
from CNN.constants import DEVICE , EPOCHS
from torchvision.models import vgg16
import torch.nn as nn
import numpy as np
from pathlib import Path
import os

In [6]:
CURRENT_DIR = Path(os.getcwd())
DATASET_DIR = CURRENT_DIR.parent / "CNN" / "dataset"

In [7]:
study = optuna.load_study(
    study_name="study7",
    storage="sqlite:///../CNN/db/optuna_study4.db"
)
model = models.AlexNet().to(DEVICE)
loss_fn=nn.CrossEntropyLoss()

lr = study.best_params["lr"]
momentum = study.best_params["momentum"]
weight_decay = study.best_params["weight_decay"]
scaling = study.best_params["scaling"]

optimizer = torch.optim.Adam(
            params= model.parameters(),
            betas=(momentum,scaling),
            lr=lr,
            weight_decay=weight_decay) \
    if study.best_params["optimizer"] == "Adam" else (torch.optim.SGD(
            params= model.parameters(),
            momentum= momentum,
            lr=lr,
))

batch_size = study.best_params["batch_size"]

transform = transforms.get_transform(study)
train_loader = data.data_loader(DATASET_DIR / "training", batch_size=batch_size, transform=transform)

test_loader = data.data_loader(DATASET_DIR / "test", batch_size=batch_size, transform=transform)


## Load the best model from the previous training

In [8]:
checkpoint = torch.load("model.pt", map_location=DEVICE)
model = models.AlexNet().to(DEVICE)

    #  Load the weights
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

## Transfer Learning with VGG16

In [9]:
vgg_model = vgg16(weights=True).to(DEVICE)
# Freeze the convolutional layers
for param in vgg_model.parameters():
    param.requires_grad = False

# Last layer from the vgg model
num_features = vgg_model.classifier[6].in_features

# Change the last layer for a new one which needs to have its weights computed
vgg_model.classifier[6] = nn.Linear(num_features, 12)


C:\Users\beni7\PycharmProjects\chrome-dino\.env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
# Get the parammeters that have the required_grad as true which are the ones to be trained
params_to_update = [p for p in vgg_model.parameters() if p.requires_grad]
# Define the optimizer
optimizer = torch.optim.Adam(params_to_update, lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
early_stopping = False
best_val_loss = np.inf

    #Training Process
for epoch in range(10):
        # Train
        train_utils.train(vgg_model, train_loader, optimizer, loss_fn, DEVICE)
        val_loss = train_utils.validation(vgg_model, test_loader, loss_fn,DEVICE)
        # Compute accuracy
        train_acc = train_utils.compute_accuracy(vgg_model, train_loader,DEVICE)
        val_acc = train_utils.compute_accuracy(vgg_model, test_loader,DEVICE)

        print(f"Epoch {epoch+1}/{10} - Train Loss: {val_loss:.6f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping = True
            # Save best model
            utils.save_checkpoint(vgg_model, study.best_params, checkpoint_path="models/transfer_model.pt")

print(f"raining completed!")

if not early_stopping:
    utils.save_checkpoint(vgg_model, study.best_params, checkpoint_path="models/transfer_model.pt")

print(f"Saved model models/transfer_model.pt")

Epoch 1/10 - Train Loss: 0.356792, Train Acc: 0.9214, Val Acc: 0.9101
Checkpoint saved to models/transfer_model.pt


In [ ]:
# Cambia el directorio de trabajo a la raíz del proyecto (assignment-2)
os.chdir("..")
os.chdir("TransferLearning")


In [ ]:
import random
from torchvision.datasets import ImageFolder


def run_evaluation(model_path, num_examples=5, device=None):
    """
    Evaluate model on random test examples.

    Loads a trained model and displays predictions vs actual labels
    for random samples from the test set.

    Args:
        model_path: Path to saved model checkpoint (e.g., "model.pt")
        num_examples: Number of examples to evaluate
        device: Device to run on (if None, uses cuda if available)
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load checkpoint
    checkpoint = torch.load(model_path, map_location="cpu")

    # Si es VGG16
    model = vgg16()
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 12)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    transform = transforms.get_transform(checkpoint["best_params"])
    test_dataset = ImageFolder(DATASET_DIR / "test", transform=transform)
    indices = random.sample(range(len(test_dataset)), num_examples)

    with torch.no_grad():
        for idx in indices:
            # Get image and true label
            original_img, label = test_dataset[idx]

            # Add batch dimension and move to device
            original_img = original_img.unsqueeze(0).to(device)

            # Forward pass
            output = model(original_img)

            # Get probabilities
            probabilities = torch.softmax(output, dim=1)

            # Get predicted class
            prediction = torch.argmax(probabilities, dim=1).item()

            print(f"Prediction: {prediction}| Actual:{label}")


In [ ]:
run_evaluation("models/transfer_model.pt",num_examples=6)
